Arithmetic Calculator Branch using RunnableBranch

In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Create model 
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Prompt for addition
add_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a calculator. Perform addition.'),
    ('human', 'Add {a} and {b}. Return only the number.')
])

# Prompt for subtraction
sub_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a calculator. Perform subtraction.'),
    ('human', 'Subtract {a} and {b}. Return only the number.')
])

# Prompt for multiplication
mul_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a calculator. Perform multiplication.'),
    ('human', 'Multiply {a} by {b}. Return only the number.')
])

# Build the branch
cal_branch = RunnableBranch(
    (lambda x: x['operation'] == 'add', add_prompt | llm | StrOutputParser()),
    (lambda x: x['operation'] == 'subtract', sub_prompt | llm | StrOutputParser()),
    (lambda x: x['operation'] == 'multiply', mul_prompt | llm | StrOutputParser()),
    add_prompt | llm | StrOutputParser()  # addition: default fallback 
)

# Test with addition
# result_add = cal_branch.invoke({
#     'operation': 'add', 
#     'a': '70',
#     'b': '35'
# })

# Define a list of input dictionaries to test all three operations
test_operations = [
    {'operation': 'add', 'a': '7', 'b': '5'},
    {'operation': 'subtract', 'a': '7', 'b': '5'},
    {'operation': 'multiply', 'a': '7', 'b': '5'}
]

for op in test_operations:
    result = cal_branch.invoke(op)
    print(f"{op['operation']} result: {result}")

add result: 12
subtract result: 2
multiply result: 35


In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch
from dotenv import load_dotenv

In [2]:
# Load environment variables
load_dotenv()

#  Create the chat model
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Create a prompt for handling refund requests
refund_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a refund specialist.'),
    ('human', 'Process this refund request: {message}')
])

# Create a prompt for handling technical issues
tech_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a technical support specialist.'),
    ('human', 'Help with this technical issue: {message}')
])

# Create a prompt for general questions (fallback)
general_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a general assistant.'),
    ('human', 'Answer this query: {message}')
])

# Build a RunnableBranch:
# It takes a list of (condition, runnable) pairs, plus a default runnable.
branch = RunnableBranch(
    (lambda x: x['type'] == 'refund', refund_prompt | llm | StrOutputParser()),
    (lambda x: x['type'] == 'technical', tech_prompt | llm | StrOutputParser()),
    general_prompt | llm | StrOutputParser()    # Default: if none of the conditions match
)

# Invoke the branch with an input dict containing 'type' and 'message'
result = branch.invoke(({
    'type': 'refund', 
    'message': 'I want a refund for buying fake laptop from your company.'
}))

print(result)

To process your refund request for the fake laptop, please follow these steps:

1. **Order Information**: Provide your order number and the date of purchase.
2. **Description of the Issue**: Briefly describe why you believe the laptop is fake and any evidence you may have (e.g., photos, discrepancies in specifications).
3. **Return Instructions**: If you have not already done so, please check our return policy for instructions on how to return the item. Typically, you will need to send the laptop back to us in its original packaging.
4. **Contact Information**: Include your contact information so we can reach you if needed.

Once we receive this information, we will review your request and initiate the refund process if it meets our policy criteria. Thank you for your cooperation!


In [4]:
# Define a list of inputs with different types
test_inputs = [
    {'type': 'refund', 'message': 'I want a refund for a laptop that arrived damaged.'},
    {'type': 'technical', 'message': 'My laptop keeps shutting down randomly.'},
    {'type': 'other', 'message': 'What is yor return policy for laptops?'}
]

# Loop through the test inputs and invoke the branch
for inp in test_inputs:
    res = branch.invoke(inp)
    print(f'Type: {inp["type"]}')
    print(res)
    print('---')

Type: refund
To process your refund request for the damaged laptop, please follow these steps:

1. **Gather Information**: Please provide the following details:
   - Order number
   - Date of purchase
   - Description of the damage
   - Photos of the damaged laptop (if available)

2. **Contact Customer Service**: Reach out to the customer service department of the retailer where you purchased the laptop. You can usually find their contact information on their website.

3. **Request a Return Authorization**: Ask for a return authorization number (RAN) or instructions on how to return the damaged laptop.

4. **Package the Laptop**: Carefully package the laptop in its original packaging, if possible, along with any accessories, manuals, and the return authorization number.

5. **Ship the Laptop**: Send the package back to the retailer using a trackable shipping method. Keep the shipping receipt for your records.

6. **Follow Up**: After the retailer receives the laptop, follow up to ensur